# FlashEats — Session 6 challenges: data profiling and validation

Yesterday we proved we could **retrieve** the data. Today's question: **can we safely use it to make a business decision?**

Validation mindset: *business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision.*

Leadership claim under review: **"Late Delivery Rate is 56 %."**

In [1]:
# --- setup: locate the repository root and load the pipeline package -------------
import sys, json, sqlite3, subprocess, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "source_systems").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
BASE = ROOT / "source_systems"            # the client systems snapshot (read-only)
NB_OUT = ROOT / "notebooks" / "output"      # scratch outputs of this notebook
NB_OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90)
print("ROOT:", ROOT, "| pack exists:", (BASE / "database" / "flasheats.db").exists())

ROOT: B:\Trimester-9\FDE\assignment-2 | pack exists: True


In [2]:
con = sqlite3.connect(f"file:{(BASE / 'database' / 'flasheats.db').as_posix()}?mode=ro", uri=True)
orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)
customers = pd.read_sql("SELECT customer_id FROM customers", con)
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")
with open(BASE / "data" / "client_metric_definitions.json") as f:
    metric_notes = json.load(f)
print("orders:", orders.shape, "| tickets:", tickets.shape, "| restaurant_status:", restaurant_status.shape)
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13) | tickets: (202, 5) | restaurant_status: (502, 4)
{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the "56 % late" claim?

Validation contract written **before** calculating anything:

| Business assumption | Data expectation | How it is tested | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` unique in `orders` | `len(orders)` vs `nunique()`; conflicting duplicates inspected | HIGH — denominator inflates |
| Delivered orders have a completion time | `final_status='delivered'` ⇒ `actual_delivery_at` not null | count nulls among delivered | HIGH — population silently shrinks, bias unknown |
| Promised ETA is valid | parseable and `promised_eta ≥ created_at` | `to_datetime(errors='coerce')`, chronology compare | HIGH — lateness against an invalid promise is meaningless |
| Event chronology is valid | `created ≤ pickup ≤ delivered` | pairwise compares | MEDIUM — stage metrics become negative |
| "Late" has an agreed definition | one owner, one formula | `client_metric_definitions.json` | HIGH — every team publishes a different number |

In [3]:
o = orders.copy()
print("Rows:", len(o), "| unique orders:", o["order_id"].nunique(), "| conflicting duplicates:", o[o["order_id"].duplicated(keep=False)]["order_id"].nunique())
print(o["final_status"].value_counts(dropna=False).to_dict())
analysis = o.drop_duplicates("order_id", keep="first").copy()
analysis["final_status_norm"] = analysis["final_status"].str.strip().str.lower()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")
checks = {
    "delivered without completion time": int(((analysis["final_status_norm"] == "delivered") & analysis["actual_delivery_at"].isna()).sum()),
    "promised_eta < created_at": int((analysis["promised_eta"] < analysis["created_at"]).sum()),
    "actual_delivery_at < pickup_at": int((analysis["actual_delivery_at"] < analysis["pickup_at"]).sum()),
    "pickup_at < created_at": int((analysis["pickup_at"] < analysis["created_at"]).sum()),
    "unparseable promised_eta": int((o.drop_duplicates("order_id")["promised_eta"].notna() & analysis["promised_eta"].isna()).sum()),
    "restaurant_id null": int(analysis["restaurant_id"].isna().sum()),
    "driver_id null": int(analysis["driver_id"].isna().sum()),
}
display(pd.Series(checks, name="violations"))
violators = analysis[(analysis["promised_eta"] < analysis["created_at"]) | (analysis["actual_delivery_at"] < analysis["pickup_at"])]
display(violators[["order_id", "created_at", "promised_eta", "pickup_at", "actual_delivery_at", "final_status"]])

Rows: 1603 | unique orders: 1600 | conflicting duplicates: 3
{'delivered': 1530, 'cancelled': 68, 'Delivered': 5}


delivered without completion time    37
promised_eta < created_at             4
actual_delivery_at < pickup_at        5
pickup_at < created_at                0
unparseable promised_eta              0
restaurant_id null                    3
driver_id null                        3
Name: violations, dtype: int64

,order_id,created_at,promised_eta,pickup_at,actual_delivery_at,final_status
50,O00051,2026-08-01 15:09:00,2026-08-01 16:28:30.928528,2026-08-01 16:24:51.495665,2026-08-01 16:19:51.495665,delivered
51,O00052,2026-08-24 17:07:00,2026-08-24 18:26:48.000000,2026-08-24 18:16:16.946011,2026-08-24 18:11:16.946011,delivered
52,O00053,2026-08-13 17:06:00,2026-08-13 17:58:50.809718,2026-08-13 18:25:53.797613,2026-08-13 18:20:53.797613,delivered
53,O00054,2026-08-28 19:56:00,2026-08-28 21:15:48.000000,2026-08-28 21:25:10.582862,2026-08-28 21:20:10.582862,delivered
54,O00055,2026-08-16 12:53:00,2026-08-16 14:05:48.000000,2026-08-16 14:08:05.645770,2026-08-16 14:03:05.645770,delivered
100,O00101,2026-08-24 20:59:00,2026-08-24 20:49:00.000000,2026-08-24 21:25:03.983984,2026-08-24 22:14:30.685278,delivered
101,O00102,2026-08-11 16:41:00,2026-08-11 16:31:00.000000,2026-08-11 17:05:05.809584,2026-08-11 17:57:17.661772,delivered
102,O00103,2026-08-22 20:11:00,2026-08-22 20:01:00.000000,2026-08-22 20:39:45.064627,2026-08-22 21:10:17.342305,delivered
103,O00104,2026-08-09 22:19:00,2026-08-09 22:09:00.000000,2026-08-09 22:47:33.934776,2026-08-09 23:36:20.740161,delivered


**Assumptions that need stakeholder clarification** (cannot be settled from the data):
- *Data Team:* why do 3 orders exist twice with different traffic buckets — two writers to the orders table?
- *Fleet Ops:* can the driver-app `delivered` event back-fill the 37 missing delivery timestamps?
- *ETA service owner:* the 4 promises made 10 minutes *before* order creation and the 5 deliveries exactly 5 minutes *before* pickup look like a systematic offset bug, not noise.
- *VP Operations:* is 'Delivered' (capitalised, 5 rows) the same status as 'delivered'? (Treated as representation.)

# Challenge 2 — Stakeholders disagree on "late"

The metric under at least three definitions, plus the validated population the pipeline publishes.

In [4]:
analysis["delay_min"] = (analysis["actual_delivery_at"] - analysis["promised_eta"]).dt.total_seconds() / 60
hist = analysis[(analysis["final_status_norm"] == "delivered") & analysis["actual_delivery_at"].notna()]
validated = hist[(hist["promised_eta"] >= hist["created_at"]) & ~(hist["actual_delivery_at"] < hist["pickup_at"])]
rows = [
    ("VP Operations — any delay > 0 min (historical population)", (hist["delay_min"] > 0).sum(), len(hist), "any missed promise counts; matches the leadership claim"),
    ("Support Lead — delay > 10 min", (hist["delay_min"] > 10).sum(), len(hist), "'meaningfully late'; hides small misses that still generate tickets"),
    ("Data Team — delivered with non-null actual time (dashboard)", (hist["delay_min"] > 0).sum(), len(hist), "population definition, same formula as VP Ops"),
    ("Finance — cancelled/refunded excluded", (hist["delay_min"] > 0).sum(), len(hist), "already true: cancelled orders never enter the population"),
    ("Pipeline — validated population (chronology rules applied)", (validated["delay_min"] > 0).sum(), len(validated), "excludes 9 orders whose timestamps contradict each other"),
]
table = pd.DataFrame(rows, columns=["definition", "late", "population", "business meaning"])
table.insert(1, "late_rate_%", (100 * table["late"] / table["population"]).round(2))
display(table)

,definition,late_rate_%,late,population,business meaning
0,VP Operations — any delay > 0 min (historical population),56.39,843,1495,any missed promise counts; matches the leadership claim
1,Support Lead — delay > 10 min,23.34,349,1495,'meaningfully late'; hides small misses that still generate tickets
2,Data Team — delivered with non-null actual time (dashboard),56.39,843,1495,"population definition, same formula as VP Ops"
3,Finance — cancelled/refunded excluded,56.39,843,1495,already true: cancelled orders never enter the population
4,Pipeline — validated population (chronology rules applied),56.33,837,1486,excludes 9 orders whose timestamps contradict each other


**Which one should leadership publish, and who must own it?** Publish the VP Operations definition (*any delivered order after the promised ETA*) on the **validated population — 56.3 %** — with the > 10 min rate (≈ 23 %) shown alongside as the severity view. It is the definition closest to the customer's experience and the one the historical dashboard already approximates (56.4 %), so nobody is surprised. **VP Operations must own it in writing**; `client_metric_definitions.json` says no canonical owner exists, so until that signature exists every publication carries the definition next to the number (gate status UNKNOWN, not PASS).

# Challenge 3 — Validate categories without cleaning by instinct

In [5]:
def inspect(name, s):
    raw = s.value_counts(dropna=False)
    norm = s.astype(str).str.strip().str.lower().str.replace(r"[\s-]+", "_", regex=True)
    print(f"--- {name}: raw values {sorted(map(repr, s.dropna().unique()))}")
    display(pd.DataFrame({"raw_count": raw}).join(pd.DataFrame({"after_representation_fix": norm.value_counts()}), how="outer").fillna(0).astype(int))

inspect("orders.final_status", orders["final_status"])
inspect("orders.traffic_bucket", orders["traffic_bucket"])
inspect("restaurant_status.status", restaurant_status["status"])
inspect("support_tickets.category", tickets["category"])

--- orders.final_status: raw values ["'Delivered'", "'cancelled'", "'delivered'"]


,raw_count,after_representation_fix
final_status,,
Delivered,5,0
cancelled,68,68
delivered,1530,1535


--- orders.traffic_bucket: raw values ["'HIGH'", "'high'", "'low'", "'medium'", "'severe'"]


,raw_count,after_representation_fix
traffic_bucket,,
HIGH,3,0
high,448,451
low,381,381
medium,635,635
severe,136,136


--- restaurant_status.status: raw values ["'READY'", "'Ready'", "'handed_off'", "'handoff'", "'preparing'", "'ready '", "'ready'", "'unknown'"]


,raw_count,after_representation_fix
status,,
READY,1,0
Ready,1,0
handed_off,168,168
handoff,1,1
preparing,171,171
ready,157,161
ready,2,0
unknown,1,1


--- support_tickets.category: raw values ["'ETA issue'", "'Late Delivery'", "'driver_not_moving'", "'eta_changed'", "'late_delivery '", "'late_delivery'", "'ready_but_waiting'", "'restaurant_delay'", "'status_mismatch'"]


,raw_count,after_representation_fix
category,,
ETA issue,1,0
Late Delivery,1,0
driver_not_moving,25,25
eta_changed,37,37
eta_issue,0,1
late_delivery,38,40
late_delivery,1,0
ready_but_waiting,33,33
restaurant_delay,37,37


| Column | Representation differences (safe to normalise) | Semantic differences (owner confirmation) |
|---|---|---|
| `final_status` | `Delivered` → `delivered` (5 rows) | none |
| `traffic_bucket` | `HIGH` → `high` (3 rows) | none in the pack; `gridlock` in the class demo would be a new bucket — ask Operations, do not map to `severe` |
| restaurant `status` | `READY`, `Ready`, `ready ` → `ready` (4 rows) | `handoff` vs `handed_off` — *probably* the same, but may be a different system writing a different milestone → kept separate, flagged (**Restaurant Ops**); `unknown` is a legitimate value the feed emits |
| ticket `category` | `Late Delivery`, `late_delivery ` → `late_delivery` | `ETA issue` vs `eta_changed` — a free-text category; kept as `eta_issue`, flagged (**Support Lead**) |

The pipeline applies exactly this split: `cleaning.py` normalises representation and records every change; `SEMANTIC_CANDIDATES` are reported, never merged.

# Challenge 4 — Cross-source integrity

In [6]:
d = analysis
rel = [
    ("orders.restaurant_id → restaurants", d["restaurant_id"].isin(restaurants["restaurant_id"]).mean(), "3 orders have a NULL restaurant_id"),
    ("orders.driver_id → drivers", d["driver_id"].isin(drivers["driver_id"]).mean(), "3 orders have a NULL driver_id"),
    ("orders.customer_id → customers", d["customer_id"].isin(customers["customer_id"]).mean(), ""),
    ("tickets.order_id → orders", tickets["order_id"].isin(d["order_id"]).mean(), "3 tickets have no order_id"),
    ("restaurant_status.order_id → orders", restaurant_status["order_id"].isin(d["order_id"]).mean(), "only 500 of 1600 orders have any status row"),
]
cov = pd.DataFrame(rel, columns=["relationship", "coverage", "note"])
cov["coverage_%"] = (100 * cov["coverage"]).round(2)
cov["status"] = cov["coverage_%"].map(lambda v: "PASS" if v == 100 else ("WARN" if v >= 95 else "FAIL"))
cov["risk"] = ["restaurant attribution impossible for those orders", "driver attribution impossible; also orders keep the ORIGINAL driver after reassignment (95 orders)", "-", "complaint cannot be reconciled with the operational story", "restaurant-side timing only known for a third of orders"]
display(cov[["relationship", "coverage_%", "status", "risk", "note"]])

,relationship,coverage_%,status,risk,note
0,orders.restaurant_id → restaurants,99.81,WARN,restaurant attribution impossible for those orders,3 orders have a NULL restaurant_id
1,orders.driver_id → drivers,99.81,WARN,driver attribution impossible; also orders keep the ORIGINAL driver after reassignment...,3 orders have a NULL driver_id
2,orders.customer_id → customers,100.00,PASS,-,
3,tickets.order_id → orders,98.51,WARN,complaint cannot be reconciled with the operational story,3 tickets have no order_id
4,restaurant_status.order_id → orders,100.00,PASS,restaurant-side timing only known for a third of orders,only 500 of 1600 orders have any status row


**If 1 % is unmapped, is that acceptable?** It depends on which decision uses those records. For the **late-delivery rate** the answer is *yes*: the KPI needs only the order's own timestamps, so the 6 orders with a missing restaurant or driver id stay in the population (flagged). For **restaurant accountability** or **driver scorecards** the answer is *no*: those 6 orders must be excluded from the ranking and the source system fixed, and the 95 reassigned orders must use Dispatch's driver, not the stale one in the orders table.

# Challenge 5 — Freshness is an SLA question

In [7]:
rs = restaurant_status.drop_duplicates().copy()          # 2 exact duplicate rows
rs["status_norm"] = rs["status"].str.strip().str.lower()
rs["last_updated_at"] = pd.to_datetime(rs["last_updated_at"], format="mixed", errors="coerce")
m = rs.merge(analysis[["order_id", "created_at", "pickup_at", "actual_delivery_at"]], on="order_id", how="left")
m["lag_vs_pickup_min"] = (m["last_updated_at"] - m["pickup_at"]).dt.total_seconds() / 60      # + = status written AFTER pickup
m["lag_vs_created_min"] = (m["last_updated_at"] - m["created_at"]).dt.total_seconds() / 60
print("orders covered:", m["order_id"].nunique(), "of", len(analysis), "| timestamp precision: minutes (no seconds)")
display(m.groupby("status_norm")["lag_vs_pickup_min"].describe().round(1))
print("ready/handed_off written more than 15 min AFTER pickup:", int((m["status_norm"].isin(["ready", "handed_off"]) & (m["lag_vs_pickup_min"] > 15)).sum()))
print("status written BEFORE the order was created:", int((m["last_updated_at"] < m["created_at"]).sum()))

orders covered: 500 of 1600 | timestamp precision: minutes (no seconds)


,count,mean,std,min,25%,50%,75%,max
status_norm,,,,,,,,
handed_off,168.0,-8.9,30.5,-368.5,-15.5,-6.4,3.0,17.6
handoff,1.0,8.1,NaN,8.1,8.1,8.1,8.1,8.1
preparing,170.0,-9.9,32.1,-380.4,-16.4,-7.3,2.0,23.7
ready,160.0,-12.9,51.9,-385.8,-17.2,-5.2,5.3,25.4
unknown,1.0,-3.2,NaN,-3.2,-3.2,-3.2,-3.2,-3.2


ready/handed_off written more than 15 min AFTER pickup: 8
status written BEFORE the order was created: 5


| Use case | Fresh enough? | Why |
|---|---|---|
| Weekly analytics (prep-time trends per restaurant) | **WARN / usable** | 31 % coverage and minute precision are tolerable for aggregates, if the coverage bias is stated |
| Live customer ETA | **FAIL** | some `ready` statuses arrive after the driver already picked up, and 5 arrive before the order exists — a live ETA built on this feed would show wrong states |
| Restaurant accountability (penalising slow kitchens) | **FAIL** | with `manual_status_updates=1` for 38 % of restaurants, a late `ready` may be a late *update*, not a late *kitchen* |

The same record is acceptable for one decision and unsafe for another — freshness is decided per use case.

# Challenge 6 — Build the validation gate

The pipeline computes this gate on every run (`output/validation_gate.md`). Reproduced here from the checks above.

In [8]:
validation_report = {
    "business_grain":        ("WARN",    "1603 rows / 1600 orders; 3 conflicting duplicates corrected (keep first) and quarantined", "Data Team to explain the double writes"),
    "timestamp_chronology":  ("WARN",    "4 promises before creation, 5 deliveries before pickup, 37 delivered without a delivery time", "exclude the 9 from the validated population; ask ETA service + Fleet Ops"),
    "kpi_definition":        ("UNKNOWN", "4 stakeholder definitions, no documented owner", "VP Operations to sign off; publish the definition next to the number"),
    "category_semantics":    ("WARN",    "representation fixed (Delivered, HIGH, READY); 'handoff' and 'ETA issue' kept separate", "Restaurant Ops / Support Lead to confirm meaning"),
    "cross_source_mapping":  ("WARN",    "99.8 % restaurant/driver coverage, 98.5 % ticket coverage, 100 % dispatch coverage; orders.driver_id stale after reassignment", "use Dispatch for the final driver; fix nulls at source"),
    "freshness":             ("WARN",    "restaurant status: 31 % coverage, minute precision, 8 stale 'ready' rows, 5 rows before creation", "weekly analytics only"),
    "publish_56_percent":    ("PUBLISH WITH CAVEATS", "56.3 % of 1 486 validated deliveries (56.4 % under the dashboard definition)", "state definition, exclusions and open owner questions with the number"),
}
display(pd.DataFrame(validation_report, index=["status", "evidence", "action"]).T)

gate_file = ROOT / "output" / "validation_gate.json"
if gate_file.exists():
    gate = json.loads(gate_file.read_text(encoding="utf-8"))
    print("Pipeline gate for the last published run:", gate["overall_status"], "→", gate["publish_decision"])

,status,evidence,action
business_grain,WARN,1603 rows / 1600 orders; 3 conflicting duplicates corrected (keep first) and quarantined,Data Team to explain the double writes
timestamp_chronology,WARN,"4 promises before creation, 5 deliveries before pickup, 37 delivered without a deliver...",exclude the 9 from the validated population; ask ETA service + Fleet Ops
kpi_definition,UNKNOWN,"4 stakeholder definitions, no documented owner",VP Operations to sign off; publish the definition next to the number
category_semantics,WARN,"representation fixed (Delivered, HIGH, READY); 'handoff' and 'ETA issue' kept separate",Restaurant Ops / Support Lead to confirm meaning
cross_source_mapping,WARN,"99.8 % restaurant/driver coverage, 98.5 % ticket coverage, 100 % dispatch coverage; or...",use Dispatch for the final driver; fix nulls at source
freshness,WARN,"restaurant status: 31 % coverage, minute precision, 8 stale 'ready' rows, 5 rows befor...",weekly analytics only
publish_56_percent,PUBLISH WITH CAVEATS,56.3 % of 1 486 validated deliveries (56.4 % under the dashboard definition),"state definition, exclusions and open owner questions with the number"


Pipeline gate for the last published run: WARN → PUBLISH WITH CAVEATS: late delivery rate = 56.33% of 1486 validated deliveries (historical definition: 56.39%). State the definition, the excluded orders and the unresolved owner questions next to the number.


**Should leadership publish "Late Delivery Rate = 56 %" today?** Yes — *with caveats and not as a bare number*: "56.3 % of validated deliveries (837 of 1 486) were delivered after the promised ETA in August; 37 deliveries have an unknown outcome and 9 were excluded for contradictory timestamps; definition owned by VP Operations (pending)". Before the metric becomes publishable **without** caveats: (1) VP Operations signs the definition, (2) Fleet Ops fixes the missing delivery timestamps or approves the driver-event back-fill, (3) the ETA service explains the negative promise offsets.

**Final takeaway:** the job was not to make the data look clean; it was to decide whether this data is safe enough for this decision, and what remains unresolved.